## 对称量化+正确率测试
1. 提取test_vector.h中的量化参数用于测试

In [1]:
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import re

# ==========================================
# 工具函数：从 C 头文件中解析参数和数组
# ==========================================
def parse_header_file(filepath):
    """从 test_vector.h 中解析出 C 常量和一维数组"""
    with open(filepath, 'r') as f:
        content = f.read()

    def get_const_val(var_type, var_name, is_float=False):
        """
        匹配类似: const int32_t L1_MULT = 104; 
        或者: const float GLOBAL_S_IN = 0.007874;
        """
        pattern = rf'const\s+{var_type}\s+{var_name}\s*=\s*([-+]?[0-9]*\.?[0-9]+([eE][-+]?[0-9]+)?);'
        match = re.search(pattern, content)
        if not match:
            raise ValueError(f"❌ 在头文件中找不到对应的 C 变量: {var_name}")
        return float(match.group(1)) if is_float else int(match.group(1))

    # 1. 解析全局 s_in
    s_in = get_const_val('float', 'GLOBAL_S_IN', is_float=True)

    # 2. 解析 4层 Conv + FC 的所有 Mult 和 Shift
    params = {
        's_in': s_in,
        'mult1': get_const_val('int32_t', 'L1_MULT'), 
        'shift1': get_const_val('int32_t', 'L1_SHIFT'),
        
        'mult2': get_const_val('int32_t', 'L2_MULT'), 
        'shift2': get_const_val('int32_t', 'L2_SHIFT'),

        'mult3': get_const_val('int32_t', 'L3_MULT'), 
        'shift3': get_const_val('int32_t', 'L3_SHIFT'),

        'mult4': get_const_val('int32_t', 'L4_MULT'), 
        'shift4': get_const_val('int32_t', 'L4_SHIFT'),
        
        'mult_fc': get_const_val('int32_t', 'FC_MULT'), 
        'shift_fc': get_const_val('int32_t', 'FC_SHIFT'),
    }

    # 3. 解析 C 数组提取纯数字
    def extract_array(array_name, dtype=torch.int8):
        pattern = r'{}\[\d+\]\s*=\s*{{([^}}]+)}}'.format(array_name)
        match = re.search(pattern, content)
        if not match:
            raise ValueError(f"❌ 找不到数组 {array_name}")
        
        num_strs = match.group(1).replace('\n', '').split(',')
        nums = [int(n.strip()) for n in num_strs if n.strip()]
        return torch.tensor(nums, dtype=dtype)

    # 提取并严格重塑为 4层 64通道网络计算所需的维度
    print("  -> 正在解析并重塑 Conv1 参数...")
    params['w1_q'] = extract_array('test_conv1_weight').view(64, 3, 3, 3)
    params['b1_q'] = extract_array('test_conv1_bias', torch.int32)
    
    print("  -> 正在解析并重塑 Conv2 参数...")
    params['w2_q'] = extract_array('test_conv2_weight').view(64, 64, 3, 3)
    params['b2_q'] = extract_array('test_conv2_bias', torch.int32)

    print("  -> 正在解析并重塑 Conv3 参数...")
    params['w3_q'] = extract_array('test_conv3_weight').view(64, 64, 3, 3)
    params['b3_q'] = extract_array('test_conv3_bias', torch.int32)

    print("  -> 正在解析并重塑 Conv4 参数...")
    params['w4_q'] = extract_array('test_conv4_weight').view(64, 64, 3, 3)
    params['b4_q'] = extract_array('test_conv4_bias', torch.int32)
    
    print("  -> 正在解析并重塑 FC 参数...")
    # 尺寸变更为: 64 * 8 * 8 = 4096
    params['wfc_q'] = extract_array('test_fc_weight').view(10, 4096)
    params['bfc_q'] = extract_array('test_fc_bias', torch.int32)

    print("  -> 头文件解析完毕！数据通道畅通。")
    return params


# ==========================================
# 主流程：在全量测试集上验证纯 INT8 逻辑
# ==========================================
def main():
    print(">>> 步骤 1/3: 加载 10000 张测试集...")
    transform = transforms.Compose([transforms.ToTensor()])
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    # 使用 batch_size=100 甚至更大，加速验证过程
    testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False)
    
    print(">>> 步骤 2/3: 从 test_vector.h 加载 NPU 量化参数...")
    try:
        hw_params = parse_header_file('test_vector.h')
    except Exception as e:
        print(f"解析头文件失败: {e}。请先运行 export_int8_params.py 生成头文件。")
        return

    s_in = hw_params['s_in']
    
    w1_q, b1_q = hw_params['w1_q'], hw_params['b1_q']
    mult1, shift1 = hw_params['mult1'], hw_params['shift1']
    
    w2_q, b2_q = hw_params['w2_q'], hw_params['b2_q']
    mult2, shift2 = hw_params['mult2'], hw_params['shift2']

    w3_q, b3_q = hw_params['w3_q'], hw_params['b3_q']
    mult3, shift3 = hw_params['mult3'], hw_params['shift3']

    w4_q, b4_q = hw_params['w4_q'], hw_params['b4_q']
    mult4, shift4 = hw_params['mult4'], hw_params['shift4']
    
    wfc_q, bfc_q = hw_params['wfc_q'], hw_params['bfc_q']
    mult_fc, shift_fc = hw_params['mult_fc'], hw_params['shift_fc']

    print("\n>>> 步骤 3/3: 在 10000 张测试集上运行 4 层纯 INT8 验证...")
    correct = 0
    total = 0

    # 整个循环内绝对不出现任何基于浮点的非线性计算！所有的运算完全模拟 FPGA/ASIC 底层！
    for images, labels in testloader:
        # 1. 输入量化 (严格使用全局 s_in)
        img_q = torch.round(images / s_in)
        img_q = torch.clamp(img_q, -127, 127).to(torch.int8)

        # 2. 模拟 NPU Conv1 -> ReLU
        mac1 = F.conv2d(img_q.float(), w1_q.float(), bias=b1_q.float(), padding=1).to(torch.int32)
        q_conv1_out = torch.clamp((mac1 * mult1) >> shift1, -127, 127).to(torch.int8)
        q_act1_out = F.relu(q_conv1_out.float()).to(torch.int8)

        # 3. 模拟 NPU Conv2 -> ReLU -> Pool1
        mac2 = F.conv2d(q_act1_out.float(), w2_q.float(), bias=b2_q.float(), padding=1).to(torch.int32)
        q_conv2_out = torch.clamp((mac2 * mult2) >> shift2, -127, 127).to(torch.int8)
        q_pool1_out = F.max_pool2d(F.relu(q_conv2_out.float()), 2).to(torch.int8)

        # 4. 模拟 NPU Conv3 -> ReLU
        mac3 = F.conv2d(q_pool1_out.float(), w3_q.float(), bias=b3_q.float(), padding=1).to(torch.int32)
        q_conv3_out = torch.clamp((mac3 * mult3) >> shift3, -127, 127).to(torch.int8)
        q_act3_out = F.relu(q_conv3_out.float()).to(torch.int8)

        # 5. 模拟 NPU Conv4 -> ReLU -> Pool2
        mac4 = F.conv2d(q_act3_out.float(), w4_q.float(), bias=b4_q.float(), padding=1).to(torch.int32)
        q_conv4_out = torch.clamp((mac4 * mult4) >> shift4, -127, 127).to(torch.int8)
        q_pool2_out = F.max_pool2d(F.relu(q_conv4_out.float()), 2).to(torch.int8)

        # 6. 模拟 NPU/CPU 协同的 FC 阶段
        q_flat_in = q_pool2_out.view(q_pool2_out.size(0), -1) 
        mac_fc = F.linear(q_flat_in.float(), wfc_q.float(), bias=bfc_q.float()).to(torch.int32)
        q_fc_out = torch.clamp((mac_fc * mult_fc) >> shift_fc, -127, 127).to(torch.int8)

        # 7. 统计正确率
        _, predicted = torch.max(q_fc_out.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"\n==================================================")
    print(f"INT8 硬件级量化模型在 10000 张测试图上的最终准确率: {accuracy:.2f}%")
    print(f"==================================================")
    print(f"(代表着烧写进 FPGA 后，硬件能跑出的真实得分！)")

if __name__ == "__main__":
    main()

>>> 步骤 1/3: 加载 10000 张测试集...
Files already downloaded and verified
>>> 步骤 2/3: 从 test_vector.h 加载 NPU 量化参数...
  -> 正在解析并重塑 Conv1 参数...
  -> 正在解析并重塑 Conv2 参数...
  -> 正在解析并重塑 Conv3 参数...
  -> 正在解析并重塑 Conv4 参数...
  -> 正在解析并重塑 FC 参数...
  -> 头文件解析完毕！数据通道畅通。

>>> 步骤 3/3: 在 10000 张测试集上运行 4 层纯 INT8 验证...

✅ INT8 硬件级量化模型在 10000 张测试图上的最终准确率: 84.96%
(这代表着你烧写进 FPGA 后，硬件能跑出的绝对真实得分！)
